# Parte 4: Integración y Concurrencia

## 4.1 Configuración del pool de conexiones

In [1]:
import threading
import time
from datetime import datetime

import pandas as pd
import psycopg2
from psycopg2 import pool, errors

# Configuren con sus credenciales
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "labdb",
    "user":     "postgres",
    "password": "postgres",
}

# Pool con minimo 2 y maximo 10 conexiones
connection_pool = pool.ThreadedConnectionPool(minconn=2, maxconn=10, **DB_CONFIG)

with connection_pool.getconn() as _c:
    with _c.cursor() as _cur:
        _cur.execute("SELECT version(), current_database();")
        _v, _db = _cur.fetchone()
connection_pool.putconn(_c)
print(_v.split(",")[0])
print("base de datos:", _db)

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu
base de datos: labdb


### Utilidades

`run_sql` toma una conexión del pool, ejecuta y la devuelve. El `try/finally`
garantiza que la conexión **siempre** vuelve al pool: si se filtrara una por
cada error, el pool se agotaría a la décima excepción y el notebook se colgaría.

In [2]:
def run_sql(sql, params=None, fetch=False, autocommit=True):
    """Ejecuta SQL con una conexion del pool y la devuelve siempre."""
    conn = connection_pool.getconn()
    try:
        conn.autocommit = autocommit
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if fetch:
                cols = [d[0] for d in cur.description]
                return pd.DataFrame(cur.fetchall(), columns=cols)
    finally:
        conn.autocommit = False
        connection_pool.putconn(conn)

## Estado inicial reproducible

In [4]:
EMPLEADOS_PRUEBA = [10001, 10002, 10003, 10004, 10005]
EMPLEADO_CONTENCION = 10010          # reservado para la demo de bloqueo


def reset_estado(empleados=None, limpiar_auditoria=True):
    """Deja a los empleados indicados como estaban antes de los ajustes de hoy."""
    emp = EMPLEADOS_PRUEBA if empleados is None else empleados
    run_sql("""
        DELETE FROM employees.salary
        WHERE employee_id = ANY(%s) AND from_date = CURRENT_DATE;
    """, (emp,))
    run_sql("""
        UPDATE employees.salary
        SET to_date = DATE '9999-01-01'
        WHERE employee_id = ANY(%s) AND to_date = CURRENT_DATE;
    """, (emp,))
    if limpiar_auditoria:
        run_sql("TRUNCATE employees.audit_salary_2025 RESTART IDENTITY;")


reset_estado(EMPLEADOS_PRUEBA + [EMPLEADO_CONTENCION])
run_sql("""
    SELECT employee_id, amount, from_date, to_date
    FROM employees.salary
    WHERE employee_id = ANY(%s) AND to_date > CURRENT_DATE
    ORDER BY employee_id;
""", (EMPLEADOS_PRUEBA,), fetch=True)

,employee_id,amount,from_date,to_date
0,10001,88958,2002-06-22,9999-01-01
1,10002,72527,2001-08-02,9999-01-01
2,10003,43311,2001-12-01,9999-01-01
3,10004,74057,2001-11-27,9999-01-01
4,10005,94692,2001-09-09,9999-01-01


`sp_ajustar_salario` inserta el nuevo salario con `from_date = CURRENT_DATE`, y
la PK de `employees.salary` es `(employee_id, from_date)`. Es decir, **un
empleado solo admite un ajuste por día**: el segundo choca contra la clave
primaria. Ese detalle es central en la simulación concurrente de 4.3, pero
también significa que sin limpiar el estado el notebook solo correría bien la
primera vez.

Esta celda deshace los ajustes de hoy sobre los empleados de prueba: borra las
filas nuevas y reabre las que el procedimiento había cerrado. Es seguro porque
el dataset original no llega más allá de 2002 (`max(from_date) = 2002-08-01`),
así que ninguna fila con fecha de hoy es del dump.

## 4.2 Función para llamar al procedimiento almacenado

Tres decisiones importantes:

- **`conn.autocommit = False`.** El procedimiento hace cuatro pasos (leer, cerrar,
  insertar, auditar). Sin transacción explícita, un fallo en el paso 4 dejaría el
  historial de salarios ya modificado y sin registro de auditoría.
- **`try / commit / except / rollback / finally putconn`.** La conexión vuelve al
  pool pase lo que pase.
- **Se clasifica el error** en lugar de solo capturarlo, para poder responder
  después qué falló y por qué. `psycopg2.errors` expone las clases SQLSTATE.

In [5]:
def ajustar_salario(employee_id: int, nuevo_salario: int, thread_id: int,
                    anio: int = 2025, hold_ms: int = 0):
    """Llama a sp_ajustar_salario desde un hilo y devuelve el resultado."""
    t0 = time.perf_counter()
    conn = connection_pool.getconn()
    res = {"hilo": thread_id, "employee_id": employee_id,
           "salario": nuevo_salario, "estado": None, "detalle": "", "ms": 0.0}
    try:
        conn.autocommit = False                      # transaccion explicita
        with conn.cursor() as cur:
            cur.execute("CALL employees.sp_ajustar_salario(%s, %s, %s);",
                        (employee_id, nuevo_salario, anio))
            # Mantiene la transaccion abierta (y por tanto el lock de fila)
            # para que la contencion entre hilos sea medible.
            if hold_ms:
                time.sleep(hold_ms / 1000)
        conn.commit()
        res["estado"] = "OK"
        res["detalle"] = "salario ajustado y auditado"

    except errors.RaiseException as e:               # trigger de validacion
        conn.rollback()
        res["estado"] = "ERROR_TRIGGER"
        res["detalle"] = str(e).splitlines()[0]

    except errors.UniqueViolation as e:              # dos ajustes el mismo dia
        conn.rollback()
        res["estado"] = "ERROR_PK"
        res["detalle"] = str(e).splitlines()[0]

    except errors.UndefinedTable as e:               # tabla de auditoria del anio
        conn.rollback()
        res["estado"] = "ERROR_TABLA"
        res["detalle"] = str(e).splitlines()[0]

    except psycopg2.Error as e:
        conn.rollback()
        res["estado"] = "ERROR_OTRO"
        res["detalle"] = f"{type(e).__name__}: {str(e).splitlines()[0]}"

    finally:
        connection_pool.putconn(conn)
        res["ms"] = round((time.perf_counter() - t0) * 1000, 1)

    return res

### Verificación secuencial

Antes de introducir concurrencia, se comprueba que el procedimiento se comporta
como se espera en los tres escenarios del enunciado.

In [6]:
reset_estado()

pruebas = [
    ("Llamada valida",                  10001, 75000, 2025),
    ("Debe fallar por trigger",         10002, 20000, 2025),
    ("Debe fallar: tabla 2099 no existe", 10003, 75000, 2099),
]

pd.DataFrame([
    dict(caso=nombre, **ajustar_salario(emp, monto, i + 1, anio=anio))
    for i, (nombre, emp, monto, anio) in enumerate(pruebas)
])[["caso", "employee_id", "salario", "estado", "detalle"]]

,caso,employee_id,salario,estado,detalle
0,Llamada valida,10001,75000,OK,salario ajustado y auditado
1,Debe fallar por trigger,10002,20000,ERROR_TRIGGER,Salario menor a 30000 no permitido
2,Debe fallar: tabla 2099 no existe,10003,75000,ERROR_TABLA,"relation ""employees.audit_salary_2099"" does no..."


El tercer caso demuestra la **atomicidad**: aunque el `UPDATE` y el `INSERT`
sobre `employees.salary` ya se habían ejecutado, el fallo del `EXECUTE` de
auditoría revierte todo. El empleado 10003 debe seguir con su salario original
y `to_date = 9999-01-01`.

In [7]:
run_sql("""
    SELECT employee_id, amount, from_date, to_date
    FROM employees.salary
    WHERE employee_id IN (10001, 10002, 10003) AND to_date > CURRENT_DATE
    ORDER BY employee_id;
""", fetch=True)

,employee_id,amount,from_date,to_date
0,10001,75000,2026-08-15,9999-01-01
1,10002,72527,2001-08-02,9999-01-01
2,10003,43311,2001-12-01,9999-01-01


## 4.3 Simulación concurrente

Seis hilos sobre cinco empleados. La lista está construida para provocar los
tres desenlaces posibles y poder responder las preguntas con evidencia:

In [ ]:
AJUSTES = [
    (10001, 75000),
    (10002, 82000),
    (10003, 25000),
    (10004, 91000),
    (10005, 68000),
    (10001, 99000),
]

resultados = []
resultados_lock = threading.Lock()


def _worker(idx, employee_id, salario):
    r = ajustar_salario(employee_id, salario, thread_id=idx + 1)
    with resultados_lock:
        resultados.append(r)


def simular_ajustes_concurrentes(n_hilos=6):
    """Lanza n_hilos hilos que intentan ajustar salarios simultaneamente."""
    reset_estado()
    resultados.clear()

    barrera = threading.Barrier(n_hilos)   # arrancan todos a la vez

    def arrancar(idx, emp, sal):
        barrera.wait()
        _worker(idx, emp, sal)

    hilos = [threading.Thread(target=arrancar, args=(i, emp, sal))
             for i, (emp, sal) in enumerate(AJUSTES[:n_hilos])]

    t0 = time.perf_counter()
    for h in hilos:
        h.start()
    for h in hilos:
        h.join()
    total = (time.perf_counter() - t0) * 1000

    print(f"{n_hilos} hilos terminados en {total:.1f} ms")
    return pd.DataFrame(sorted(resultados, key=lambda r: r["hilo"]))


df_sim = simular_ajustes_concurrentes(n_hilos=6)
df_sim

6 hilos terminados en 58.0 ms


,hilo,employee_id,salario,estado,detalle,ms
0,1,10001,75000,ERROR_PK,duplicate key value violates unique constraint...,48.0
1,2,10002,82000,OK,salario ajustado y auditado,48.0
2,3,10003,25000,ERROR_TRIGGER,Salario menor a 30000 no permitido,47.9
3,4,10004,91000,OK,salario ajustado y auditado,47.9
4,5,10005,68000,OK,salario ajustado y auditado,55.5
5,6,10001,99000,OK,salario ajustado y auditado,47.9


Los seis hilos arrancan a la vez con una `threading.Barrier`. Cuál de los dos
hilos que compiten por el 10001 confirma y cuál es rechazado **no es
determinista**: depende de cuál llegue primero al `UPDATE`. Lo que sí está
garantizado es que exactamente uno gana.

### Resumen por desenlace

In [9]:
df_sim.groupby("estado").agg(
    hilos=("hilo", "count"),
    empleados=("employee_id", lambda s: sorted(set(s))),
    ms_medio=("ms", "mean"),
).reset_index()

,estado,hilos,empleados,ms_medio
0,ERROR_PK,1,[10001],48.000
1,ERROR_TRIGGER,1,[10003],47.900
2,OK,4,"[10001, 10002, 10004, 10005]",49.825


### Contención sobre el mismo empleado, en cámara lenta

En la simulación anterior los seis hilos terminan en decenas de milisegundos, así
que la espera por el lock existe pero no se distingue del ruido. Para medirla hay
que forzar el orden:

In [10]:
def demo_contencion(employee_id=EMPLEADO_CONTENCION, hold_ms=400, retraso_b_ms=100):
    # empleado propio y sin tocar la auditoria, para no alterar la simulacion anterior
    reset_estado([employee_id], limpiar_auditoria=False)
    salida = {}

    def hilo_a():
        salida["A"] = ajustar_salario(employee_id, 70000, 1, hold_ms=hold_ms)

    def hilo_b():
        time.sleep(retraso_b_ms / 1000)      # deja que A tome el lock primero
        salida["B"] = ajustar_salario(employee_id, 80000, 2)

    a, b = threading.Thread(target=hilo_a), threading.Thread(target=hilo_b)
    a.start(); b.start()
    a.join();  b.join()

    df = pd.DataFrame([salida["A"], salida["B"]], index=["A (retiene el lock)",
                                                         "B (llega despues)"])
    # B arranca cuando A ya lleva retraso_b_ms de sus hold_ms retenidos,
    # asi que deberia quedarse bloqueado alrededor de (hold_ms - retraso_b_ms).
    ref = df_sim["ms"].median()          # llamada tipica sin competencia
    print(f"A retuvo la transaccion {hold_ms} ms y confirmo en {salida['A']['ms']:.0f} ms")
    print(f"B tardo {salida['B']['ms']:.0f} ms; sin competencia una llamada "
          f"tarda ~{ref:.0f} ms  ->  ~{salida['B']['ms'] - ref:.0f} ms bloqueado")
    return df[["employee_id", "salario", "estado", "ms", "detalle"]]


demo_contencion()

A retuvo la transaccion 400 ms y confirmo en 405 ms
B tardo 306 ms; sin competencia una llamada tarda ~48 ms  ->  ~258 ms bloqueado


,employee_id,salario,estado,ms,detalle
A (retiene el lock),10010,70000,OK,405.2,salario ajustado y auditado
B (llega despues),10010,80000,ERROR_PK,305.7,duplicate key value violates unique constraint...


Se usa un empleado aparte (10010) para no alterar lo que dejó la simulación:

- **Hilo A** ajusta al empleado 10010 y **retiene la transacción 400 ms** antes
  del `COMMIT`, manteniendo el bloqueo sobre la fila.
- **Hilo B** arranca 100 ms más tarde, cuando A ya tiene el lock, e intenta
  ajustar al mismo empleado.

Si el bloqueo de fila existe, B no puede fallar de inmediato: tiene que quedarse
esperando hasta que A confirme, es decir, los ~300 ms de retención que le quedan.
Si no existiera, B fallaría en el mismo tiempo que cualquier otra llamada.

## 4.4 Verificación de auditoría

In [11]:
def ver_auditoria():
    conn = connection_pool.getconn()
    try:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT employee_id, old_amount, new_amount, changed_at
                FROM employees.audit_salary_2025
                ORDER BY changed_at DESC
                LIMIT 20;
            """)
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]

        df = pd.DataFrame(rows, columns=cols)
        display(df)
    finally:
        connection_pool.putconn(conn)


ver_auditoria()

,employee_id,old_amount,new_amount,changed_at
0,10010,80324,70000,2026-08-15 23:23:20.878600+00:00
1,10005,94692,68000,2026-08-15 23:23:20.811855+00:00
2,10002,72527,82000,2026-08-15 23:23:20.777391+00:00
3,10001,88958,99000,2026-08-15 23:23:20.764229+00:00
4,10004,74057,91000,2026-08-15 23:23:20.764159+00:00


La auditoría contiene **solo los ajustes que se confirmaron**. Los hilos que
fallaron (trigger y clave primaria) no dejaron rastro: su `ROLLBACK` deshizo
también el `INSERT` de auditoría. Esto confirma que la tabla de auditoría y el
historial de salarios no pueden desincronizarse.

Y el estado final de `employees.salary` debe tener **exactamente una fila
vigente por empleado**, aunque dos hilos hayan intentado ajustar al 10001:

In [12]:
run_sql("""
    SELECT employee_id,
           count(*) FILTER (WHERE to_date > CURRENT_DATE) AS filas_vigentes,
           max(amount) FILTER (WHERE to_date > CURRENT_DATE) AS salario_vigente
    FROM employees.salary
    WHERE employee_id = ANY(%s)
    GROUP BY employee_id
    ORDER BY employee_id;
""", (EMPLEADOS_PRUEBA + [EMPLEADO_CONTENCION],), fetch=True)

,employee_id,filas_vigentes,salario_vigente
0,10001,1,99000
1,10002,1,82000
2,10003,1,43311
3,10004,1,91000
4,10005,1,68000
5,10010,1,70000


## Preguntas 4

### 1. ¿Qué pasaría si no usaran `conn.autocommit = False` y uno de los pasos del procedimiento fallara a mitad de camino?

En realidad hay que separar dos niveles. `CALL` de un procedimiento PL/pgSQL ya
es atómico *dentro del servidor*: si el `EXECUTE` de auditoría falla, PostgreSQL
aborta el bloque entero y revierte el `UPDATE` y el `INSERT` previos. Eso se
comprobó arriba con el empleado 10003, que quedó intacto tras el fallo de la
tabla `audit_salary_2099`.

Lo que `autocommit = False` protege es el nivel del **cliente**: con
`autocommit = True` cada sentencia que envía psycopg2 se confirma por separado,
de modo que no se puede agrupar el `CALL` con ninguna otra operación en la misma
unidad de trabajo, ni decidir desde Python abortar lo ya hecho. En cuanto la
transacción incluya algo más (una segunda llamada, una inserción en otra tabla,
una validación posterior) el fallo dejaría un estado parcial confirmado e
irreversible: salarios cerrados sin su reemplazo, o ajustes sin auditoría.
Además, sin transacción explícita el `rollback()` del manejador de excepciones no
tendría nada que revertir y el `except` sería puramente decorativo.

### 2. ¿Algún hilo obtuvo un error por validación del trigger? ¿Qué salario lo causó?

Sí. El **hilo 3**, sobre el empleado **10003**, con un salario de **25 000**, por
debajo del mínimo de 30 000. El error es
`ERROR_TRIGGER: Salario menor a 30000 no permitido`, lanzado por
`employees.fn_validar_salario()` en el paso 3 del procedimiento (el `INSERT`
sobre `employees.salary`). Los pasos 1 y 2 ya se habían ejecutado, pero el
`ROLLBACK` los deshizo: el empleado 10003 conserva su salario original.

Es un `BEFORE INSERT`, así que el trigger rechaza la fila **antes** de escribirla:
no hay que deshacer ninguna escritura física, solo abortar la transacción.

### 3. ¿Qué ventaja ofrece el `ThreadedConnectionPool` frente a abrir y cerrar una conexión nueva en cada hilo?

Abrir una conexión a PostgreSQL no es barato: implica handshake TCP, autenticación
`scram-sha-256` y que el servidor haga `fork` de un proceso backend
dedicado. Son decenas de milisegundos por conexión, que con seis hilos dominarían
el tiempo total y ahogarían justamente lo que se quiere medir.

El pool crea `minconn=2` conexiones al arrancar y las reutiliza: `getconn()` y
`putconn()` son operaciones de memoria. Además `maxconn=10` actúa como **límite
de admisión**: si llegaran 500 peticiones simultáneas, el pool las hace esperar en
vez de abrir 500 backends y tumbar el servidor por agotamiento de `max_connections`.
La variante *Threaded* añade los locks internos que hacen seguro compartir el pool
entre hilos, algo que `SimpleConnectionPool` no garantiza.

El precio es la disciplina del `finally`: una conexión que no se devuelve queda
perdida, y a la décima el pool se bloquea para siempre.

### 4. Si dos hilos intentan ajustar al **mismo empleado** al mismo tiempo, ¿qué mecanismo de PostgreSQL evita la corrupción de datos? ¿Observaron algún comportamiento de espera?

Los hilos 1 y 6 compiten por el empleado 10001 y la salida confirma que uno hace `OK` y el otro recibe `ERROR_PK`; nunca quedan dos filas vigentes. Actúan dos mecanismos encadenados: primero, el `UPDATE ... WHERE employee_id = X AND to_date > CURRENT_DATE` del procedimiento toma un bloqueo de fila, de modo que el primer hilo retiene el lock exclusivo hasta el `COMMIT` y el segundo simplemente espera. Esa espera no se distingue en la simulación de seis hilos porque las transacciones son muy cortas, pero la celda en cámara lenta (reteniendo 400 ms) muestra que el hilo B tarda unos 300 ms. Al desbloquearse bajo `READ COMMITTED`, PostgreSQL reevalúa el `UPDATE`: la fila ya no cumple `to_date > CURRENT_DATE` porque el ganador la cerró, afecta 0 filas y el perdedor intenta insertar una fila duplicada; la restricción `UNIQUE` de la PK lo rechaza con `UniqueViolation`.

El segundo mecanismo es el que realmente salva los datos. El bloqueo serializa, pero por sí solo no bastaría: sin la restricción `UNIQUE`, el hilo perdedor insertaría una segunda fila vigente para el mismo empleado y corrompería el `AVG` de las consultas. Conviene notar que el rechazo por PK es un efecto secundario afortunado del diseño, no una regla de negocio explícita: `sp_ajustar_salario` no contempla ajustar dos veces el mismo día a un empleado. Si ese caso fuera legítimo habría que decidirlo explícitamente con `SELECT ... FOR UPDATE` y actualizar la fila del día en vez de insertar otra. La corrección depende de una restricción declarada en el esquema; la base de datos es el lugar correcto para hacer cumplir esas reglas.

## Cierre

In [13]:
connection_pool.closeall()
print("pool cerrado")

pool cerrado
